# Lab 3 · Live data you can trust

**In class:** Part A, guided, on the World Bank API · **At home:** Part B, the same steps on an API you might use for your project
**How it fits:** this optional notebook shows each pipeline step one at a time. Use it if you want to see what happens inside `pipeline.py`. Labs 3 + 4 are due **Sunday, November 22**.

Tonight's four skills:
1. **Fetch** live data from an API, page by page
2. **Flatten** the response into a table and load it into DuckDB
3. **Check** it against five rules that define "ready", and stop if it fails
4. **Share** the validated table in the cloud, on MotherDuck

Run cells top to bottom with **Shift+Enter**.

---
# Part A · Guided: urban population from the World Bank

The question: **what share of each country's people live in cities, and how is that changing?** The World Bank publishes this as indicator `SP.URB.TOTL.IN.ZS`, "Urban population (% of total population)".

## A1. One request

An API is a web address that returns data instead of a web page. The address has three parts:

- **Base:** `https://api.worldbank.org/v2`
- **Endpoint** (what you're asking for): `/country/all/indicator/SP.URB.TOTL.IN.ZS`
- **Parameters** (options, after the `?`): `format=json`, which years, how many rows per page

In [1]:
%pip install "duckdb==1.5.6"


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import time
from pathlib import Path
import requests
import pandas as pd
import duckdb

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent

BASE = "https://api.worldbank.org/v2"
INDICATOR = "SP.URB.TOTL.IN.ZS"            # urban population, % of total
URL = f"{BASE}/country/all/indicator/{INDICATOR}"
PARAMS = {"format": "json", "date": "2015:2023", "per_page": 1000}

response = requests.get(URL, params={**PARAMS, "page": 1}, timeout=90)
print("status:", response.status_code)    # 200 means OK
response.raise_for_status()               # stop here if the request failed

meta, records = response.json()           # the World Bank returns [page info, rows]
meta

status: 200


{'page': 1,
 'pages': 3,
 'per_page': 1000,
 'total': 2394,
 'sourceid': '2',
 'lastupdated': '2026-11-11'}

`meta` is **information about the data**: which page this is, how many pages there are, and the total number of rows. One record looks like this:

In [3]:
records[0]

{'indicator': {'id': 'SP.URB.TOTL.IN.ZS',
  'value': 'Urban population (% of total population)'},
 'country': {'id': 'ZH', 'value': 'Africa Eastern and Southern'},
 'countryiso3code': 'AFE',
 'date': '2023',
 'value': 37.7723012408349,
 'unit': '',
 'obs_status': '',
 'decimal': 0}

**Notice:** the record is **nested**, like last week's document example. `country` is a little object inside the row, with its own `id` and `value`. We'll flatten it into plain columns.

## A2. Fetch every page, politely

APIs hand out data in pages. We ask for page 1, 2, 3… until we reach the last page, pausing briefly between requests so we don't overload the server (that's what **rate limits** protect against).

In [4]:
def fetch_all_pages(url, params, pause=0.5):
    rows, page = [], 1
    while True:
        r = requests.get(url, params={**params, "page": page}, timeout=90)
        r.raise_for_status()
        meta, records = r.json()
        rows.extend(records or [])
        print(f"page {meta['page']} of {meta['pages']}: {len(records or [])} rows")
        if page >= meta["pages"]:
            return rows, meta
        page += 1
        time.sleep(pause)                 # be polite between requests

raw_rows, meta = fetch_all_pages(URL, {**PARAMS, "per_page": 500})
print("rows received:", len(raw_rows), " rows the API says exist:", meta["total"])

page 1 of 5: 500 rows
page 2 of 5: 500 rows
page 3 of 5: 500 rows
page 4 of 5: 500 rows
page 5 of 5: 394 rows
rows received: 2394  rows the API says exist: 2394


**Rows in should equal rows out**, again. If those two numbers differ, a page went missing.

## A3. Flatten into a table

`pd.json_normalize` turns nested records into columns: `country` → `{id, value}` becomes `country.id` and `country.value`. Then we keep and rename what we need.

In [5]:
urban = pd.json_normalize(raw_rows)
urban = urban.rename(columns={
    "countryiso3code": "country_code",
    "country.value": "country",
    "date": "year",
    "value": "urban_pct",
})[["country_code", "country", "year", "urban_pct"]]

urban["year"] = urban["year"].astype(int)
urban.head()

,country_code,country,year,urban_pct
0,AFE,Africa Eastern and Southern,2023,37.772301
1,AFE,Africa Eastern and Southern,2022,37.360578
2,AFE,Africa Eastern and Southern,2021,36.908543
3,AFE,Africa Eastern and Southern,2020,36.488322
4,AFE,Africa Eastern and Southern,2019,36.097331


## A4. Countries vs. regions: a join from last week

The World Bank mixes **regions** ("World", "Sub-Saharan Africa", "High income") in with countries. If we average across all rows, regions get counted as if they were countries. A second endpoint lists every code with its region, and regions are labelled `Aggregates`. Join, then keep real countries only.

In [8]:
from datetime import datetime

backup_name = f"lab3_backup_{datetime.now():%Y%m%d_%H%M%S}.duckdb"
for suffix in ("", ".wal"):
    old_file = ROOT / f"lab3.duckdb{suffix}"
    if old_file.exists():
        old_file.rename(ROOT / f"{backup_name}{suffix}")
        print(f"Backed up {old_file.name} as {backup_name}{suffix}")

Backed up lab3.duckdb as lab3_backup_20261123_104901.duckdb
Backed up lab3.duckdb.wal as lab3_backup_20261123_104901.duckdb.wal


In [9]:
countries_raw, _ = fetch_all_pages(f"{BASE}/country", {"format": "json", "per_page": 100})
countries = pd.json_normalize(countries_raw).rename(columns={
    "id": "country_code", "region.value": "region", "incomeLevel.value": "income_level",
})[["country_code", "region", "income_level"]]

print("codes in the lookup:", len(countries))
countries["region"].value_counts().head(8)

page 1 of 3: 100 rows
page 2 of 3: 100 rows
page 3 of 3: 96 rows
codes in the lookup: 296


region
Aggregates                                           79
Europe & Central Asia                                58
Sub-Saharan Africa                                   48
Latin America & Caribbean                            42
East Asia & Pacific                                  37
Middle East, North Africa, Afghanistan & Pakistan    23
South Asia                                            6
North America                                         3
Name: count, dtype: int64

In [10]:
con = duckdb.connect(str(ROOT / "lab3.duckdb"))
con.sql("CREATE OR REPLACE TABLE urban_raw AS SELECT * FROM urban")
con.sql("CREATE OR REPLACE TABLE countries AS SELECT * FROM countries")

con.sql("""
    CREATE OR REPLACE TABLE urban_clean AS
    SELECT u.country_code, u.country, c.region, c.income_level, u.year, u.urban_pct
    FROM urban_raw u
    JOIN countries c ON u.country_code = c.country_code
    WHERE trim(c.region) <> 'Aggregates'   -- trim: some region names end in a space
""")

con.sql("""
    SELECT (SELECT count(*) FROM urban_raw)   AS rows_from_api,
           (SELECT count(*) FROM urban_clean) AS rows_for_countries
""")

┌───────────────┬────────────────────┐
│ rows_from_api │ rows_for_countries │
│     int64     │       int64        │
├───────────────┼────────────────────┤
│          2394 │               1953 │
└───────────────┴────────────────────┘

**What happened to the rest?** Fewer rows is expected here: the regions were removed on purpose. That's the difference between losing rows by accident (last week) and dropping them by decision (this week). Write the decision down.

## A5. Five checks that define "ready"

Before anyone uses this table, it has to pass five rules. Each check is a SQL query that counts **problem rows**; zero problems means pass.

In [11]:
class ValidationError(Exception):
    pass

def run_checks(con, table, checks):
    """Run each check; print PASS/FAIL; stop loudly if any fail."""
    failures = []
    for name, sql in checks:
        problems = con.execute(sql.format(t=table)).fetchall()[0][0]
        status = "PASS" if problems == 0 else "FAIL"
        print(f"{status}  {name}  ({problems} problem rows)")
        if problems:
            failures.append(name)
    if failures:
        raise ValidationError(f"{table} is NOT ready: {failures}")
    print(f"\n{table} is ready.")

CHECKS = [
    # 1. Completeness of the download: did every page arrive?
    ("all pages arrived",
     f"SELECT abs((SELECT count(*) FROM urban_raw) - {meta['total']})"),
    # 2. Keys: every row has a country and a year, and no country-year appears twice
    ("keys present and unique",
     "SELECT count(*) FROM (SELECT country_code, year, count(*) n FROM {t} "
     "GROUP BY ALL HAVING n > 1 OR country_code IS NULL OR year IS NULL)"),
    # 3. Validity: a percentage must be between 0 and 100
    ("urban_pct between 0 and 100",
     "SELECT count(*) FROM {t} WHERE urban_pct < 0 OR urban_pct > 100"),
    # 4. Completeness of values: no more than 25% of values missing
    ("under 25% of values missing",
     "SELECT CASE WHEN avg(CASE WHEN urban_pct IS NULL THEN 1 ELSE 0 END) > 0.25 THEN 1 ELSE 0 END FROM {t}"),
    # 5. Freshness: there is data for 2022 or later
    ("has data from 2022 or later",
     "SELECT CASE WHEN max(year) FILTER (WHERE urban_pct IS NOT NULL) >= 2022 THEN 0 ELSE 1 END FROM {t}"),
]

run_checks(con, "urban_clean", CHECKS)

PASS  all pages arrived  (0 problem rows)
PASS  keys present and unique  (0 problem rows)
PASS  urban_pct between 0 and 100  (0 problem rows)
PASS  under 25% of values missing  (0 problem rows)
PASS  has data from 2022 or later  (0 problem rows)

urban_clean is ready.


## A6. What failure looks like

Real data breaks. Here we copy the table and plant one impossible value: a country that's 150% urban. Watch the checks catch it and **stop**.

In [12]:
con.execute("CREATE OR REPLACE TABLE urban_broken AS SELECT * FROM urban_clean")
con.execute("UPDATE urban_broken SET urban_pct = 150 WHERE country_code = 'KEN' AND year = 2020")

try:
    run_checks(con, "urban_broken", CHECKS)
except ValidationError as e:
    print("\nSTOPPED:", e)
    print("Nothing was written to the cloud. The last good table is still there.")

PASS  all pages arrived  (0 problem rows)
PASS  keys present and unique  (0 problem rows)
FAIL  urban_pct between 0 and 100  (1 problem rows)
PASS  under 25% of values missing  (0 problem rows)
PASS  has data from 2022 or later  (0 problem rows)

STOPPED: urban_broken is NOT ready: ['urban_pct between 0 and 100']
Nothing was written to the cloud. The last good table is still there.


That's what **failing loudly** means: the bad data never reaches anyone downstream, and the message says exactly which rule broke. In Session 4 this same function guards an automatic, daily pipeline.

## A7. Share it: write to MotherDuck

Your `lab3.duckdb` file lives only in this Codespace. **MotherDuck** is DuckDB in the cloud: a table there can be reached by a scheduled job or a web app.

**Before running this cell**, you need your MotherDuck token saved as a Codespaces secret named `MOTHERDUCK_TOKEN` (we do this together in class; steps are in `labs/lab3.md`). The code never contains the token: DuckDB reads it from the secret automatically.

In [13]:
import os

# GitHub stores secret names in capitals; DuckDB looks for either spelling. This line covers both.
token = os.environ.get("MOTHERDUCK_TOKEN") or os.environ.get("motherduck_token")
if token:
    os.environ["motherduck_token"] = token

if not token:
    print("No MotherDuck token found. Follow 'Set up your MotherDuck token' in labs/lab3.md,")
    print("then restart your Codespace and run this cell again.")
else:
    run_checks(con, "urban_clean", CHECKS)          # never publish unchecked data
    to_share = con.sql("SELECT * FROM urban_clean").df()

    md = duckdb.connect("md:")                       # 'md:' means MotherDuck
    md.sql("CREATE DATABASE IF NOT EXISTS course")
    md.register("to_share", to_share)
    md.sql("CREATE OR REPLACE TABLE course.main.urban_population AS SELECT * FROM to_share")
    print(md.sql("SELECT count(*) AS rows_in_cloud FROM course.main.urban_population"))

PASS  all pages arrived  (0 problem rows)
PASS  keys present and unique  (0 problem rows)
PASS  urban_pct between 0 and 100  (0 problem rows)
PASS  under 25% of values missing  (0 problem rows)
PASS  has data from 2022 or later  (0 problem rows)

urban_clean is ready.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────┐
│ rows_in_cloud │
│     int64     │
├───────────────┤
│          1953 │
└───────────────┘



Open **app.motherduck.com** in your browser: you should see a database called `course` with a table `urban_population`.

## A8. Use it

Which countries urbanized fastest between 2015 and 2023? This query runs on the validated data.

In [14]:
con.sql("""
    SELECT country, region,
           max(urban_pct) FILTER (WHERE year = 2015) AS pct_2015,
           max(urban_pct) FILTER (WHERE year = 2023) AS pct_2023,
           round(pct_2023 - pct_2015, 1)              AS change
    FROM urban_clean
    GROUP BY country, region
    HAVING pct_2015 IS NOT NULL AND pct_2023 IS NOT NULL
    ORDER BY change DESC
    LIMIT 10
""")

┌────────────────────┬────────────────────────────┬──────────────────┬──────────────────┬────────┐
│      country       │           region           │     pct_2015     │     pct_2023     │ change │
│      varchar       │          varchar           │      double      │      double      │ double │
├────────────────────┼────────────────────────────┼──────────────────┼──────────────────┼────────┤
│ Thailand           │ East Asia & Pacific        │ 46.9521043776171 │ 60.9341926574707 │   14.0 │
│ Rwanda             │ Sub-Saharan Africa         │ 18.1953149858416 │ 28.8555310645477 │   10.7 │
│ Cambodia           │ East Asia & Pacific        │ 30.0669222549819 │ 40.6290788072508 │   10.6 │
│ Kiribati           │ East Asia & Pacific        │ 50.9541970402064 │ 61.5244676194408 │   10.6 │
│ Somalia, Fed. Rep. │ Sub-Saharan Africa         │ 43.8618107408479 │ 54.0714435506882 │   10.2 │
│ Switzerland        │ Europe & Central Asia      │ 75.6044960143169 │ 85.1854212679815 │    9.6 │
│ Kosovo  

---
# Part B · At home: an API you might use for your project

Pick a public API you're considering for your final project (your dataset choice is due **November 15**, with this lab). Many of the portals from Lab 1 have one:

- **NYC Open Data** (and most city portals): every dataset page has an **API** button. URLs look like `https://data.cityofnewyork.us/resource/xxxx-xxxx.json`
- **World Bank:** any indicator, using the same code as Part A
- Others: data.gov, Census, transit agencies. Prefer APIs that **don't need a key**.

## B1. One request

In [15]:
import json
import requests

MY_URL = "https://api.worldbank.org/v2/country/all/indicator/SP.POP.GROW"
MY_PARAMS = {"format": "json", "date": "2015:2023", "per_page": 500}

r = requests.get(MY_URL, params={**MY_PARAMS, "page": 1}, timeout=90)
print("status:", r.status_code)
r.raise_for_status()

my_meta, my_records = r.json()
print("Request URL:", r.url)
print(f"Page {my_meta['page']} of {my_meta['pages']}")

my_sample = next(row for row in my_records if row["value"] is not None)
print(json.dumps(my_sample, indent=2, ensure_ascii=False))

status: 200
Request URL: https://api.worldbank.org/v2/country/all/indicator/SP.POP.GROW?format=json&date=2015%3A2023&per_page=500&page=1
Page 1 of 5
{
  "indicator": {
    "id": "SP.POP.GROW",
    "value": "Population growth (annual %)"
  },
  "country": {
    "id": "ZH",
    "value": "Africa Eastern and Southern"
  },
  "countryiso3code": "AFE",
  "date": "2023",
  "value": 2.51916664518144,
  "unit": "",
  "obs_status": "",
  "decimal": 1
}


**Describe the response** (double-click to edit): Is it a list of records, or page info plus records (like the World Bank)? Which fields are nested? What will one row of your table be (the grain)?

## B2. Fetch at least two pages

Pick the pattern your API uses. **NYC Open Data and other Socrata portals** use `$limit` and `$offset`; the World Bank uses `page`.

The JSON response contains page info and records, with nested indicator and country fields. After filtering out aggregates, each row gives one country’s annual population growth rate for one year.

In [16]:
def fetch_socrata(url, params=None, page_size=1000, max_pages=5, pause=0.5):
    """NYC Open Data style: ask for page_size rows at a time, skipping ahead with $offset."""
    rows = []
    for i in range(max_pages):
        p = {**(params or {}), "$limit": page_size, "$offset": i * page_size}
        r = requests.get(url, params=p, timeout=90)
        r.raise_for_status()
        batch = r.json()
        print(f"page {i + 1}: {len(batch)} rows")
        rows.extend(batch)
        if len(batch) < page_size:    # a short page means we've reached the end
            break
        time.sleep(pause)
    return rows

my_rows, my_meta = fetch_all_pages(MY_URL, MY_PARAMS)
print("Rows received:", len(my_rows))
print("Rows expected:", my_meta["total"])

page 1 of 5: 500 rows
page 2 of 5: 500 rows
page 3 of 5: 500 rows
page 4 of 5: 500 rows
page 5 of 5: 394 rows
Rows received: 2394
Rows expected: 2394


## B3. Flatten, clean, load

In [17]:
# Flatten nested JSON and keep the fields needed for country-year comparisons.
mine = pd.json_normalize(my_rows).rename(columns={
    "countryiso3code": "country_code",
    "country.value": "country",
    "date": "year",
    "value": "population_growth_pct",
})[["country_code", "country", "year", "population_growth_pct"]].copy()

# Use numeric types for year and growth so SQL can compare them.
mine["year"] = pd.to_numeric(mine["year"]).astype("Int64")
mine["population_growth_pct"] = pd.to_numeric(mine["population_growth_pct"])

# Use Part A's country lookup to exclude regional and income-group aggregates.
con.execute("""
    CREATE OR REPLACE TABLE my_data AS
    SELECT m.*
    FROM mine AS m
    WHERE EXISTS (
        SELECT 1
        FROM countries AS c
        WHERE c.country_code = m.country_code
          AND trim(c.region) <> 'Aggregates'
    )
""")

# Confirm that the table loaded and has the expected column types.
print("Rows loaded:", con.execute("SELECT count(*) FROM my_data").fetchall()[0][0])
con.execute("DESCRIBE my_data").fetchdf()

Rows loaded: 1953


,column_name,column_type,null,key,default,extra
0,country_code,VARCHAR,YES,None,None,None
1,country,VARCHAR,YES,None,None,None
2,year,BIGINT,YES,None,None,None
3,population_growth_pct,DOUBLE,YES,None,None,None


## B4. Your five checks

Write five checks that define "ready" for **your** data. Each is a SQL query that returns the number of problem rows (`{t}` stands for the table name). Aim for one of each kind: completeness, keys, validity, allowed values, freshness.

In [18]:
MY_CHECKS = [
    ("1. Completeness: at least 75% of growth rates are present",
     "SELECT CASE WHEN count(*) > 0 "
     "AND count(population_growth_pct) >= 0.75 * count(*) "
     "THEN 0 ELSE 1 END FROM {t}"),

    ("2. Keys: country and year are present and unique together",
     "SELECT count(*) FROM ("
     "SELECT country_code, year FROM {t} "
     "GROUP BY country_code, year "
     "HAVING count(*) > 1 OR country_code IS NULL "
     "OR trim(country_code) = '' OR year IS NULL"
     ") AS bad_keys"),

    ("3. Validity: reported growth rates are finite",
     "SELECT count(*) FROM {t} "
     "WHERE population_growth_pct IS NOT NULL "
     "AND NOT isfinite(population_growth_pct)"),

    ("4. Allowed values: all years are within 2015–2023",
     "SELECT count(*) FROM {t} WHERE year NOT BETWEEN 2015 AND 2023"),

    ("5. Freshness: at least one growth rate is available for 2023",
     "SELECT CASE WHEN count(*) > 0 THEN 0 ELSE 1 END FROM {t} "
     "WHERE year = 2023 AND population_growth_pct IS NOT NULL"),
]

run_checks(con, "my_data", MY_CHECKS)

PASS  1. Completeness: at least 75% of growth rates are present  (0 problem rows)
PASS  2. Keys: country and year are present and unique together  (0 problem rows)
PASS  3. Validity: reported growth rates are finite  (0 problem rows)
PASS  4. Allowed values: all years are within 2015–2023  (0 problem rows)
PASS  5. Freshness: at least one growth rate is available for 2023  (0 problem rows)

my_data is ready.


**Why these five?** For each check, one sentence: what would go wrong downstream if it failed?

1. Too many missing growth rates would make country comparisons less representative. 
2. Missing keys would leave records unmatched, and duplicate keys could multiply rows in a join.
3. Non-finite growth rates could invalidate averages and charts.
4. Years outside the requested window could distort the comparison with Part A.
5. Without a 2023 observation, the table would miss the final year of the comparison.

## B5. Write it to MotherDuck

In [19]:
MY_TABLE = "my_project_data"    # <- a short name for your table

run_checks(con, "my_data", MY_CHECKS)              # only publish data that passes
os.environ.setdefault("motherduck_token", os.environ.get("MOTHERDUCK_TOKEN", ""))
md = duckdb.connect("md:")
md.sql("CREATE DATABASE IF NOT EXISTS course")
md.register("to_share", con.sql("SELECT * FROM my_data").df())
md.sql(f"CREATE OR REPLACE TABLE course.main.{MY_TABLE} AS SELECT * FROM to_share")
print(md.sql(f"SELECT count(*) AS rows_in_cloud FROM course.main.{MY_TABLE}"))

PASS  1. Completeness: at least 75% of growth rates are present  (0 problem rows)
PASS  2. Keys: country and year are present and unique together  (0 problem rows)
PASS  3. Validity: reported growth rates are finite  (0 problem rows)
PASS  4. Allowed values: all years are within 2015–2023  (0 problem rows)
PASS  5. Freshness: at least one growth rate is available for 2023  (0 problem rows)

my_data is ready.
┌───────────────┐
│ rows_in_cloud │
│     int64     │
├───────────────┤
│          1953 │
└───────────────┘



## B6. Next: put it in your pipeline

You don't need to copy any Python. Your pipeline is set up in one settings file, `project.toml`:

- the API address and columns go in **[source]**
- your checks from B4 go in **[[checks]]** (the same SQL, with `raw` as the table name)
- the table you want your app to use goes in **[[tables]]**

Run `python pipeline.py --preview` in the terminal to see your API's columns. `examples/project_worldbank.toml` shows a World Bank version.

**Never commit your token.** It lives only in your secrets.

In [21]:
for path in (ROOT / "examples" / "project_worldbank.toml", ROOT / "project.toml"):
    print(f"\n--- {path.name} ---")
    print(path.read_text())


--- project_worldbank.toml ---
# =============================================================================
#  Example settings for a World Bank indicator.
#  To use it: copy this file over project.toml
#     cp examples/project_worldbank.toml project.toml
#  then change the indicator code. Find codes at data.worldbank.org: open an
#  indicator page; the code is in its address (e.g. SP.URB.TOTL.IN.ZS).
#
#  The World Bank API is slow (a run can take 3 to 5 minutes). That's normal.
#  The pipeline removes regions and income groups, so you get countries only.
#  Your raw table has: country_code, country, year, value, region, income_level
# =============================================================================

[project]
name = "urban_population"

[source]
type = "worldbank"
indicator = "SP.URB.TOTL.IN.ZS"     # urban population, % of total population
years = "2015:2023"

[[checks]]
name = "enough rows arrived"
why  = "about 200 countries x 9 years; far fewer means pages went m